# L10 — Introduction to SimPy and the simdes Package

**Module**: M04 | **Chapter**: 6 | **Lecture**: L10

## Learning Objectives
By the end of this notebook you will be able to:
1. Write a SimPy process as a Python generator.
2. Model a single-server queue using `simpy.Resource`.
3. Install and import the `simdes` package; use `MM1Queue`.
4. Verify simulation output against M/M/1 analytical benchmarks.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"SimPy version: {simpy.__version__}")

## 1. SimPy Hello World — A Barista Queue

Three core concepts:
- `simpy.Environment()` — the clock and event calendar
- `env.timeout(t)` — wait `t` simulated time units
- `simpy.Resource(env, capacity=c)` — a limited server

In [ ]:
def customer(env, name, barista, rng, mu, records):
    """A single customer: arrive, wait, get served, depart."""
    arrival = env.now
    with barista.request() as req:
        yield req                                     # wait for barista
        wait = env.now - arrival
        svc = rng.exponential(1.0 / mu)              # service time
        yield env.timeout(svc)                        # serve customer
    records.append({'name': name, 'arrival': arrival,
                    'wait': wait, 'sojourn': wait + svc})

def arrivals(env, barista, rng, lam, mu, records):
    """Generate customers at Poisson rate lam."""
    i = 0
    while True:
        yield env.timeout(rng.exponential(1.0 / lam))
        env.process(customer(env, f'C{i}', barista, rng, mu, records))
        i += 1

# --- Run the simulation ---
LAM, MU, SIM_TIME = 3.0, 4.0, 10_000.0
rng = np.random.default_rng(42)
env = simpy.Environment()
barista = simpy.Resource(env, capacity=1)
records = []

env.process(arrivals(env, barista, rng, LAM, MU, records))
env.run(until=SIM_TIME)

df = pd.DataFrame(records)
print(f"Customers served : {len(df)}")
print(f"Simulated  Wq    : {df['wait'].mean():.4f}")
print(f"Theoretical Wq   : {LAM / (MU * (MU - LAM)):.4f}")

## 2. Monitoring Queue Length Over Time

In [ ]:
def queue_monitor(env, resource, snapshots, interval=1.0):
    """Sample queue length (waiting, not in service) at fixed intervals."""
    while True:
        snapshots.append((env.now, len(resource.queue)))
        yield env.timeout(interval)

rng2 = np.random.default_rng(7)
env2 = simpy.Environment()
server2 = simpy.Resource(env2, capacity=1)
records2 = []
snapshots = []

env2.process(arrivals(env2, server2, rng2, LAM, MU, records2))
env2.process(queue_monitor(env2, server2, snapshots, interval=0.5))
env2.run(until=200.0)

snap_df = pd.DataFrame(snapshots, columns=['time', 'queue_len'])

fig, ax = plt.subplots(figsize=(10, 3))
ax.step(snap_df['time'], snap_df['queue_len'], where='post', color='steelblue', lw=0.8)
ax.set_xlabel('Simulated time')
ax.set_ylabel('Queue length (waiting)')
ax.set_title('M/M/1 queue length over time (λ=3, μ=4)')
plt.tight_layout()
plt.show()

print(f"Time-avg queue length (sim): {snap_df['queue_len'].mean():.4f}")
rho = LAM / MU
print(f"Theory Lq = ρ²/(1-ρ)      : {rho**2 / (1 - rho):.4f}")

## 3. Using the simdes Package

In [ ]:
from simdes.models.queues import MM1Queue

model = MM1Queue(
    arrival_rate=3.0,
    service_rate=4.0,
    sim_time=50_000.0,
    seed=0,
)
result = model.run()
print("Single run result:")
for k, v in result.items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k:25s}: {v}")

print(f"\nTheoretical Wq : {model.theoretical_mean_wait_queue():.4f}")
print(f"Theoretical ρ  : {model.theoretical_utilization():.4f}")

In [ ]:
# Run 30 replications
rep_df = model.run_replications(n=30)
print(rep_df[['mean_wait_queue', 'utilization']].describe().round(4))

# 95% CI
from scipy.stats import t as t_dist
col = 'mean_wait_queue'
m = rep_df[col].mean()
se = rep_df[col].std() / np.sqrt(len(rep_df))
hw = t_dist.ppf(0.975, df=len(rep_df)-1) * se
print(f"\n95% CI for Wq: ({m-hw:.4f}, {m+hw:.4f})")
print(f"Theory Wq    : {model.theoretical_mean_wait_queue():.4f}")

## 4. Little's Law Verification

In [ ]:
# Little's Law: L = lambda * W
# Check: mean_queue_length ≈ lam * mean_wait_queue
mean_L = rep_df['mean_queue_length'].mean()
mean_Wq = rep_df['mean_wait_queue'].mean()
lam_check = mean_L / mean_Wq

print(f"Mean Lq (sim)      : {mean_L:.4f}")
print(f"Mean Wq (sim)      : {mean_Wq:.4f}")
print(f"Lq / Wq (= λ)      : {lam_check:.4f}   (theory λ=3.0)")

---
## Try It Yourself

1. Increase `capacity=2` in the barista simulation and re-run. Is the simulated `Wq` closer to the M/M/1 or M/M/2 analytical value? (Compute the M/M/2 Erlang-C result by hand using Chapter 7.)

2. Add a `simpy.Container` to track the total number of customers who have arrived (incrementing by 1 at each arrival). After `env.run`, compare this to `len(records)`. Why might they differ?

3. Run `MM1Queue` with `arrival_rate=3.9, service_rate=4.0` (ρ=0.975). How long a `sim_time` do you need to get a 95% CI half-width below 0.5 on `mean_wait_queue`?